# 7-3 문자 단위 RNN: 실습 2개

[07-03 문자 단위 RNN(Char RNN): 실습 2개](https://wikidocs.net/64703)

RNN의 입출력의 단위가 단어 레벨(word-level)이 아니라 문자 레벨(character-level)로 하여 RNN을 구현하면, 이를 문자 단위 RNN이라 함. 문자 단위 RNN을 다대다 구조로 구현.

## 1. 문자 단위 RNN (Char RNN)

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

### 1) 훈련 데이터 전처리하기
문자 시퀀스 'apple' 입력받으면 pple!를 출력하는 RNN 구현.

입력 데이터와 레이블 데이터에 대해서 문자 집합(vocabulary)을 만듬. 여기서 문자 집합은 중복을 제거한 문자들의 집합.

In [2]:
input_str = 'apple'
label_str = 'pple!'
char_vocab = sorted(list(set(input_str + label_str)))
vocab_size = len(char_vocab)
print('문자 집합의 크기: {}'.format(vocab_size))

문자 집합의 크기: 5


현재 문자 집합에는 총 5개의 문자가 있음. !, a, e, l, p임.

하이퍼파라미터 정의. 이때 입력은 원-핫 벡터를 사용할 것이므로 입력의 크기는 문자 집합의 크기여야만 함.

In [3]:
input_size = vocab_size
hidden_size = 5
output_size = 5
learning_rate = 0.1

In [4]:
# 문자 집합에 고유한 정수 부여
char_to_index = dict((c, i) for i, c in enumerate(char_vocab))
print(char_to_index)

{'!': 0, 'a': 1, 'e': 2, 'l': 3, 'p': 4}


- !은 0, a는 1, e는 2, l은 3, p는 4가 부여

나중에 예측 결과를 다시 문자 시퀀스로 보기 위해 반대로 정수로부터 문자를 얻을 수 있는 index_to_char 만듦.

In [5]:
index_to_char = {}

for key, value in char_to_index.items():
  index_to_char[value] = key
print(index_to_char)

{0: '!', 1: 'a', 2: 'e', 3: 'l', 4: 'p'}


입력 데이터와 레이블 데이터의 각 문자들을 정수로 맵핑

In [6]:
x_data = [char_to_index[c] for c in input_str]
y_data = [char_to_index[c] for c in label_str]
print(x_data)
print(y_data)

[1, 4, 4, 3, 2]
[4, 4, 3, 2, 0]


파이토치의 nn.RNN()은 기본적으로 3차원 텐서를 입력받음. 그렇기 때문에 배치 차원을 추가해줌.

In [7]:
# 배치 차원 추가
# 텐서 연산인 unsqueeze(0)를 통해 해결할 수도 있었음
x_data = [x_data]
y_data = [y_data]

print(x_data)
print(y_data)

[[1, 4, 4, 3, 2]]
[[4, 4, 3, 2, 0]]


입력 시퀀스의 각 문자들을 원-핫 벡터로 바꿔줌

In [8]:
x_one_hot = [np.eye(vocab_size)[x] for x in x_data]
print(x_one_hot)

[array([[0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0.]])]


입력 데이터와 레이블 데이터를 텐서로 바꿔줌

In [9]:
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)

/tmp/ipython-input-2348034151.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  X = torch.FloatTensor(x_one_hot)


In [10]:
# 각 텐서의 크기 확인
print('훈련 데이터의 크기: {}'.format(X.shape))
print('레이블의 크기: {}'.format(Y.shape))

훈련 데이터의 크기: torch.Size([1, 5, 5])
레이블의 크기: torch.Size([1, 5])


### 2) 모델 구현하기

In [13]:
class Net(torch.nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super(Net, self).__init__()
    self.rnn = torch.nn.RNN(input_size, hidden_size, batch_first=True)
    self.fc = torch.nn.Linear(hidden_size, output_size, bias=True)

  def forward(self, x):
    x, _status = self.rnn(x)
    x = self.fc(x)
    return x

In [14]:
net = Net(input_size, hidden_size, output_size)

In [15]:
outputs = net(X)
print(outputs.shape)

torch.Size([1, 5, 5])


(1,5,5)의 크기를 가지는데 각각 배치 차원, 시점, 출력의 크기임. 나중에 정확도를 측정할 때는 이를 모두 펼쳐서 계산하게 되는데, 이때는 view를 사용하여 배치 차원과 시점 차원을 하나로 만듦.

In [16]:
print(outputs.view(-1, input_size).shape)

torch.Size([5, 5])


차원이 (5,5)가 됨. 이제 레이블 데이터의 크기를 다시 복습.

In [17]:
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([1, 5])
torch.Size([5])


레이블 데이터는 (1,5)의 크기를 갖는데, 마찬가지로 나중에 정확도를 측정할 때 이걸 펼쳐서 계산함. 이 경우 (5)의 크기를 가지게 됨.

옵티마이저와 손실 함수 정의

In [18]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

In [19]:
# 총 100번의 에포크 학습
for i in range(100):
  optimizer.zero_grad()
  outputs = net(X)
  loss = criterion(outputs.view(-1, input_size), Y.view(-1)) # view하는 이유는 batch 차원 제거 위해
  loss.backward() # 기울기 계산
  optimizer.step() # 아까 optimizer 선언 시 넣어둔 파라미터 업데이트

  # 아래 세 줄은 실제 어떻게 예측했는지 확인하기 위한 코드
  result = outputs.data.numpy().argmax(axis=2)
  result_str = ''.join([index_to_char[c] for c in np.squeeze(result)])
  print(i, 'loss: ', loss.item(), 'prediction: ', result, 'true Y: ',
        y_data, 'prediction str: ', result_str)

0 loss:  1.8239656686782837 prediction:  [[1 1 4 4 1]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  aappa
1 loss:  1.5126245021820068 prediction:  [[4 4 4 4 4]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  ppppp
2 loss:  1.3522679805755615 prediction:  [[4 4 4 4 4]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  ppppp
3 loss:  1.2453092336654663 prediction:  [[4 4 4 4 4]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  ppppp
4 loss:  1.1576083898544312 prediction:  [[4 4 3 4 4]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pplpp
5 loss:  1.0308067798614502 prediction:  [[4 4 3 4 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pplp!
6 loss:  0.8342680931091309 prediction:  [[4 4 3 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pple!
7 loss:  0.6606022715568542 prediction:  [[4 4 3 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pple!
8 loss:  0.5231879353523254 prediction:  [[4 4 3 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pple!
9 loss:  0.39868536591529846 prediction:  [[4 4 3 2 0]]

## 2. 더 많은 데이터로 학습한 문자 단위 RNN

In [20]:
import torch
import torch.nn as nn
import torch.optim as optim

In [21]:
sentence = ("if you want to build a ship, don't drum up people together to"
"collect wood and don't assign them tasks and work, but rather"
"teach them to long for the endless immensity of the sea.")

In [23]:
char_set = list(set(sentence))
char_dic = { c:i for i, c in enumerate(char_set)}

In [25]:
print(char_dic)

{'w': 0, 'n': 1, 'l': 2, 't': 3, 'e': 4, 'k': 5, 'd': 6, 'f': 7, 'p': 8, 'y': 9, 'a': 10, 's': 11, 'm': 12, '.': 13, 'i': 14, ',': 15, 'c': 16, "'": 17, 'o': 18, ' ': 19, 'h': 20, 'r': 21, 'b': 22, 'g': 23, 'u': 24}


각 문자에 정수가 부여되었으며, 총 25개의 문자 존재. 문자 집합의 크기 확인

In [26]:
dic_size = len(char_dic)
print('문자 집합의 크기: {}'.format(dic_size))

문자 집합의 크기: 25


문자 집합의 크기는 25이며, 입력을 원-핫 벡터로 사용할 것이므로 이는 매 시점마다 들어갈 입력의 크기이기도 함. 이제 하이퍼파라미터를 설정함. hidden_size를 입력의 크기와 동일하게 줬는데, 이는 사용자의 선택으로 다른 값을 줘도 무방.

sequence_length라는 변수를 선언했는데, 앞서 만든 샘플을 10개 단위로 끊어서 샘플을 만들 예정이기 때문.

In [27]:
# 하이퍼파라미터 설정
hidden_size = dic_size
sequence_length = 10
learning_rate = 0.1

In [28]:
x_data = []
y_data = []

for i in range(0, len(sentence) - sequence_length):
  x_str = sentence[i : i+sequence_length]
  y_str = sentence[i+1 : i+sequence_length+1]
  print(i, x_str, '->', y_str)

  x_data.append([char_dic[c] for c in x_str])
  y_data.append([char_dic[c] for c in y_str])

0 if you wan -> f you want
1 f you want ->  you want 
2  you want  -> you want t
3 you want t -> ou want to
4 ou want to -> u want to 
5 u want to  ->  want to b
6  want to b -> want to bu
7 want to bu -> ant to bui
8 ant to bui -> nt to buil
9 nt to buil -> t to build
10 t to build ->  to build 
11  to build  -> to build a
12 to build a -> o build a 
13 o build a  ->  build a s
14  build a s -> build a sh
15 build a sh -> uild a shi
16 uild a shi -> ild a ship
17 ild a ship -> ld a ship,
18 ld a ship, -> d a ship, 
19 d a ship,  ->  a ship, d
20  a ship, d -> a ship, do
21 a ship, do ->  ship, don
22  ship, don -> ship, don'
23 ship, don' -> hip, don't
24 hip, don't -> ip, don't 
25 ip, don't  -> p, don't d
26 p, don't d -> , don't dr
27 , don't dr ->  don't dru
28  don't dru -> don't drum
29 don't drum -> on't drum 
30 on't drum  -> n't drum u
31 n't drum u -> 't drum up
32 't drum up -> t drum up 
33 t drum up  ->  drum up p
34  drum up p -> drum up pe
35 drum up pe -> rum up peo
36

각 샘플의 각 문자들은 고유한 정수로 인코딩된 상태.

In [29]:
print(x_data[0]) # if you wan
print(y_data[0]) # f you want

[14, 7, 19, 9, 18, 24, 19, 0, 10, 1]
[7, 19, 9, 18, 24, 19, 0, 10, 1, 3]


한 번씩 쉬프트 된 시퀀스가 정상적으로 출력되는 것을 볼 수 있음. 입력 시퀀스에 대해 원-핫 인코딩을 수행하고, 입력 데이터와 레이블 데이터를 텐서로 변환함.

In [31]:
x_one_hot = [np.eye(dic_size)[x] for x in x_data]
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)

In [32]:
print('훈련 데이터의 크기: {}'.format(X.shape))
print('레이블의 크기: {}'.format(Y.shape))

훈련 데이터의 크기: torch.Size([168, 10, 25])
레이블의 크기: torch.Size([168, 10])


In [33]:
print(X[0])

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,

In [34]:
print(Y[0])

tensor([ 7, 19,  9, 18, 24, 19,  0, 10,  1,  3])


### 2) 모델 구현하기
- 은닉층 2개

In [35]:
class Net(torch.nn.Module):
  def __init__(self, input_dim, hidden_dim, layers):
    super(Net, self).__init__()
    self.rnn = torch.nn.RNN(input_dim, hidden_dim, num_layers=layers, batch_first=True)
    self.fc = torch.nn.Linear(hidden_dim, hidden_dim, bias=True)

  def forward(self, x):
    x, _status = self.rnn(x)
    x = self.fc(x)
    return x

In [36]:
net = Net(dic_size, hidden_size, 2)

- nn.RNN() 안에 num_layers라는 인자 사용
  - 이는 은닉층을 몇 개 쌓을 것인지 의미

In [37]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

In [38]:
outputs = net(X)
print(outputs.shape)

torch.Size([168, 10, 25])


torch.Size([168, 10, 25]) => 각각 배치 차원, 시점, 출력의 크기. 나중에 정확도를 측정할 때는 이를 모두 펼쳐서 계산하게 되는데, 이때는 view를 사용하여 배치 차원, 시점 차원을 하나로 만듦

In [39]:
print(outputs.view(-1, dic_size).shape)

torch.Size([1680, 25])


In [40]:
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([168, 10])
torch.Size([1680])


In [41]:
for i in range(100):
  optimizer.zero_grad()
  outputs = net(X)
  loss = criterion(outputs.view(-1, dic_size), Y.view(-1))
  loss.backward()
  optimizer.step()

  results = outputs.argmax(dim=2)
  predict_str = ""
  for j, result in enumerate(results):
    if j == 0:
      predict_str += ''.join([char_set[t] for t in result])

    else:
      predict_str += char_set[result[-1]]

  print(predict_str)

es,i,,bele,beebe,e,,eee,le,,bee,,lele,lb,,eb,le,,eebee,,,,lbleee,pe,be,eeeee,ebeb,leee,ee,,ee,l,eee,,bee,ee,e,ebe,,bel,i,,,,le,bb,,l,elbe,eeee,,bbe,,e,,,,ebbe,,i,,be,,ee,ee,l,ll
                                                                                                                                                                                 
eeleeele alllllll enllalalalalalllalalalalclaalalalelalelalalllellalalalll a llalalalalglelneeelelal llallellllalalllalalalal  ele  llall llalallllellllalela elalallla elale lll
  t t t    t  t     t t t   t        t t tt     t         t  t    t t        t  tt     t tt  t t  tt   t       t        t    t t t  t  m  t t        t     tt      t  t  t       
p ......., gr. g ., . bhg hg ,hg  .  ..t.ghg hg gghg . h, g bb, ..hg  .t .h, . bt ..g ,. e, ...th. g .  gt.b, ..,b...h,. g. gh,t, h.t..hge.g g. g.e, ., hg .t.b. b . gb  t. .  .h
s wgow os  s   s o  s   se   sos o   s    os o  s o,o  o  s       os   o  osos     o  se   o    o os   o,     

처음에는 이상한 예측을 하지만 마지막 에포크에서는 꽤 정확한 문자를 생성하는 것을 볼 수 있음.